# Fire detector v2: train on several datasets, then compare with v1

Run the cells **one at a time, top to bottom**. Each step says what to expect. If a cell shows a red error, copy the last 10 lines and send them to Claude.

**What this notebook does**
1. Downloads fire/smoke pictures from several open datasets (FASDD, D-Fire, a small drone set).
2. Puts them on **one class list: fire and smoke**, and keeps the pictures *without* fire as "negatives" so the model learns what is not fire.
3. Fine-tunes your existing v1 model on the mix (not from scratch, so it is much shorter).
4. **Compares v1 and v2 on the same pictures**, source by source, and suggests a confidence threshold.
5. Exports the Raspberry Pi version and runs the live laptop-camera test.

**Before you start**
- Menu: **Runtime → Change runtime type → T4 GPU → Save**.
- Have your Roboflow **Private API key** ready. Paste it only into this notebook, never into a chat.
- Your v1 model should be in Drive at `fire_yolo/fire_smoke_yolov8n/weights/best.pt`.

**Honest limits.** The comparison uses held-out pictures from the same datasets, which flatters any model. The real test is your own clips and pictures (Step 10).

## Step 1. Check the GPU and install the tools

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

Expect a table that mentions **Tesla T4** (or another GPU). If it says *command not found*, you forgot the GPU setting above.

## Step 2. Connect Google Drive and find your v1 model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/fire_yolo'
os.makedirs(SAVE_DIR, exist_ok=True)
V1_WEIGHTS = f'{SAVE_DIR}/fire_smoke_yolov8n/weights/best.pt'
RUN_NAME = 'fire_v2_yolov8n'
NEW_WEIGHTS = f'{SAVE_DIR}/{RUN_NAME}/weights/best.pt'
MERGED_DIR = '/content/merged_fire'
MERGED_YAML = f'{MERGED_DIR}/data.yaml'
print('v1 model found:', os.path.exists(V1_WEIGHTS), '|', V1_WEIGHTS)

Expect `v1 model found: True`. If it says False, training will start from the small generic `yolov8n.pt` instead (slower to reach the same quality) and the v1-vs-v2 comparison will be skipped.

## Step 3. Install the dataset-merging helper
This cell only writes a file next to the notebook. Nothing is downloaded yet.

In [ ]:
%%writefile merge_datasets.py
"""
Merge several YOLO-format fire/smoke datasets into one training set
----------------------------------------------------------------------------
Every source dataset names its classes differently (FASDD lost its names and
only has '0' and '1'; D-Fire has an extra 'object' class). This module maps
every source onto ONE class list:

        0 = fire      1 = smoke

and drops everything else. Pictures whose boxes were all dropped (or that had
no boxes to begin with) are KEPT as empty-label pictures: those are the
"hard negatives" that teach the detector what is NOT fire.

Every output file name starts with the source name (e.g. dfire__img123.jpg) so
results can later be broken down per source.

Used by train_fire_detector_v2.ipynb. Self-test:  python merge_datasets.py
"""

from __future__ import annotations

import glob
import os
import random
import shutil
from typing import Dict, List, Optional

import yaml

CLASS_NAMES = ["fire", "smoke"]
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
SPLIT_ALIASES = {"train": ("train",), "valid": ("valid", "val", "validation")}


# ---------------------------------------------------------------------------
# Class mapping
# ---------------------------------------------------------------------------

def read_names(root: str) -> List[str]:
    """Class names from root/data.yaml, as a list ordered by class id."""
    cfg = yaml.safe_load(open(os.path.join(root, "data.yaml")))
    names = cfg.get("names", [])
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names, key=lambda x: int(x))]
    return [str(n) for n in names]


def auto_map(names: List[str]) -> Optional[Dict[int, Optional[int]]]:
    """Guess source-class -> {0 fire, 1 smoke, None drop} from the NAMES.
    Returns None when the names say nothing (e.g. '0','1'): the caller must
    then give the mapping by hand after looking at pictures."""
    if all(n.strip().isdigit() for n in names):
        return None
    out: Dict[int, Optional[int]] = {}
    for i, n in enumerate(names):
        low = n.lower()
        if "smoke" in low:
            out[i] = 1
        elif "fire" in low or "flame" in low:
            out[i] = 0
        else:
            out[i] = None
    if 0 not in out.values():
        return None          # no class we can call fire: refuse to guess
    return out


# ---------------------------------------------------------------------------
# Label conversion
# ---------------------------------------------------------------------------

def convert_label_text(text: str, class_map: Dict[int, Optional[int]]):
    """Returns (new_text, kept_per_class{0,1}, n_dropped). Handles box lines
    ('c x y w h') and polygon lines ('c x1 y1 x2 y2 ...', turned into a box)."""
    kept = {0: 0, 1: 0}
    dropped = 0
    lines = []
    for line in text.splitlines():
        p = line.split()
        if len(p) < 5:
            continue
        try:
            c = int(float(p[0]))
            vals = [float(v) for v in p[1:]]
        except ValueError:
            continue
        dst = class_map.get(c)
        if dst is None:
            dropped += 1
            continue
        if len(vals) == 4:
            x, y, w, h = vals
        else:                                   # polygon -> bounding box
            xs, ys = vals[0::2], vals[1::2]
            x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
            x, y, w, h = (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0
        if w <= 0 or h <= 0:
            dropped += 1
            continue
        x, y = min(max(x, 0.0), 1.0), min(max(y, 0.0), 1.0)
        w, h = min(w, 1.0), min(h, 1.0)
        lines.append(f"{dst} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
        kept[dst] += 1
    return ("\n".join(lines) + ("\n" if lines else ""), kept, dropped)


# ---------------------------------------------------------------------------
# Finding pictures
# ---------------------------------------------------------------------------

def split_dirs(root: str, split: str):
    """(images_dir, labels_dir) for 'train' or 'valid', or None if absent."""
    for alias in SPLIT_ALIASES[split]:
        im = os.path.join(root, alias, "images")
        if os.path.isdir(im):
            return im, os.path.join(root, alias, "labels")
    return None


def list_pairs(images_dir: str, labels_dir: str):
    pairs = []
    for p in sorted(glob.glob(os.path.join(images_dir, "*"))):
        if os.path.splitext(p)[1].lower() not in IMG_EXTS:
            continue
        stem = os.path.splitext(os.path.basename(p))[0]
        lp = os.path.join(labels_dir, stem + ".txt")
        pairs.append((p, lp if os.path.exists(lp) else None))
    return pairs


def _kept_boxes(lp, class_map):
    if lp is None:
        return 0
    k = convert_label_text(open(lp).read(), class_map)[1]
    return k[0] + k[1]


# ---------------------------------------------------------------------------
# Merge
# ---------------------------------------------------------------------------

def merge(sources: List[dict], out_dir: str, seed: int = 1) -> dict:
    """sources: dicts with
         name, root, class_map (None -> auto_map), train_cap, val_cap (None = all),
         max_empty_frac (largest share of empty pictures in what is taken, e.g. 0.4)
       Writes out_dir/{train,valid}/{images,labels} and out_dir/data.yaml.
       Returns statistics per source and split."""
    rng = random.Random(seed)
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    stats: dict = {}
    for s in sources:
        name, root = s["name"], s["root"]
        cmap = s.get("class_map")
        if cmap is None:
            cmap = auto_map(read_names(root))
            if cmap is None:
                raise ValueError(f"{name}: class names are not informative; give class_map by hand "
                                 f"(look at pictures with preview_grid first)")
        cmap = {int(k): v for k, v in cmap.items()}
        for split in ("train", "valid"):
            d = split_dirs(root, split)
            st = stats.setdefault(name, {}).setdefault(
                split, {"images": 0, "empty": 0, "fire_boxes": 0, "smoke_boxes": 0, "dropped_boxes": 0})
            if d is None:
                continue
            pairs = list_pairs(*d)
            pos = [p for p in pairs if _kept_boxes(p[1], cmap) > 0]
            emp = [p for p in pairs if _kept_boxes(p[1], cmap) == 0]
            rng.shuffle(pos)
            rng.shuffle(emp)
            cap = s.get(f"{'train' if split == 'train' else 'val'}_cap")
            if cap is None:
                chosen = pos + emp
                mef = s.get("max_empty_frac")
                if mef is not None and pos:
                    # at most mef of the final set is empty: e <= mef/(1-mef) * len(pos)
                    emp_cap = int(round(mef / (1 - mef) * len(pos))) if mef < 1 else len(emp)
                    chosen = pos + emp[:emp_cap]
            else:
                mef = s.get("max_empty_frac")
                n_emp = min(len(emp), int(cap * mef) if mef is not None else len(emp))
                n_pos = min(len(pos), cap - n_emp)
                n_emp = min(len(emp), cap - n_pos)     # give unused room back to empties
                chosen = pos[:n_pos] + emp[:n_emp]
            oi = os.path.join(out_dir, split, "images")
            ol = os.path.join(out_dir, split, "labels")
            os.makedirs(oi, exist_ok=True)
            os.makedirs(ol, exist_ok=True)
            for ip, lp in chosen:
                base = f"{name}__{os.path.basename(ip)}"
                stem = os.path.splitext(base)[0]
                shutil.copy(ip, os.path.join(oi, base))
                if lp is not None:
                    txt, kept, dropped = convert_label_text(open(lp).read(), cmap)
                else:
                    txt, kept, dropped = "", {0: 0, 1: 0}, 0
                open(os.path.join(ol, stem + ".txt"), "w").write(txt)
                st["images"] += 1
                st["empty"] += int(kept[0] + kept[1] == 0)
                st["fire_boxes"] += kept[0]
                st["smoke_boxes"] += kept[1]
                st["dropped_boxes"] += dropped
    cfg = {"path": os.path.abspath(out_dir), "train": "train/images", "val": "valid/images",
           "nc": 2, "names": CLASS_NAMES}
    yaml.safe_dump(cfg, open(os.path.join(out_dir, "data.yaml"), "w"))
    return stats


def print_stats(stats: dict):
    print(f"{'source':<10}{'split':<7}{'images':>8}{'empty':>8}{'fire boxes':>12}{'smoke boxes':>13}{'dropped':>9}")
    tot = {"train": 0, "valid": 0}
    for name, sp in stats.items():
        for split, st in sp.items():
            print(f"{name:<10}{split:<7}{st['images']:>8}{st['empty']:>8}{st['fire_boxes']:>12}"
                  f"{st['smoke_boxes']:>13}{st['dropped_boxes']:>9}")
            tot[split] += st["images"]
    print(f"TOTAL train {tot['train']}, valid {tot['valid']}")


# ---------------------------------------------------------------------------
# Look at labels (do this once per source with unknown class order)
# ---------------------------------------------------------------------------

def preview_grid(root: str, class_map: Optional[dict], n: int = 8, seed: int = 1, split: str = "train"):
    """Draws n random pictures with their boxes. Colour = ORIGINAL class id of the
    source (0 orange, 1 blue, 2 green, 3 magenta, others grey) so you can see which
    id sits on flames. Returns a matplotlib figure."""
    import cv2
    import matplotlib.pyplot as plt
    d = split_dirs(root, split)
    pairs = [p for p in list_pairs(*d) if p[1] and os.path.getsize(p[1]) > 0]
    random.Random(seed).shuffle(pairs)
    cols = {0: (255, 140, 0), 1: (0, 120, 255), 2: (0, 200, 0), 3: (220, 0, 220)}
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for ax, (ip, lp) in zip(axes.ravel(), pairs[:n]):
        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        seen = set()
        for line in open(lp):
            p = line.split()
            if len(p) < 5:
                continue
            c = int(float(p[0]))
            vals = [float(v) for v in p[1:]]
            if len(vals) == 4:
                x, y, bw, bh = vals
            else:
                xs, ys = vals[0::2], vals[1::2]
                x, y, bw, bh = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)
            x1, y1, x2, y2 = int((x - bw / 2) * w), int((y - bh / 2) * h), int((x + bw / 2) * w), int((y + bh / 2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), cols.get(c, (128, 128, 128)), max(2, w // 300))
            seen.add(c)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title("source classes in picture: " + ",".join(map(str, sorted(seen))), fontsize=9)
    fig.suptitle("orange = source class 0, blue = 1, green = 2, magenta = 3", fontsize=12)
    return fig


# ---------------------------------------------------------------------------
# Self-test
# ---------------------------------------------------------------------------

def _selftest():
    import tempfile
    import numpy as np
    import cv2

    # --- auto_map
    assert auto_map(["fire", "smoke"]) == {0: 0, 1: 1}
    assert auto_map(["smoke", "fire"]) == {0: 1, 1: 0}
    assert auto_map(["fire", "smoke", "object"]) == {0: 0, 1: 1, 2: None}
    assert auto_map(["0", "1"]) is None                     # unnamed: must be given by hand
    assert auto_map(["smoke", "person"]) is None            # nothing fire-like: refuse to guess
    assert auto_map(["Fire-Flame"]) == {0: 0}

    # --- label conversion
    txt = "0 0.5 0.5 0.2 0.2\n1 0.3 0.3 0.1 0.1\n2 0.1 0.1 0.1 0.1\n0 0.5 0.5 0 0.2\n"
    out, kept, dropped = convert_label_text(txt, {0: 0, 1: 1, 2: None})
    assert kept == {0: 1, 1: 1} and dropped == 2 and out.count("\n") == 2
    out, kept, _ = convert_label_text(txt, {0: 1, 1: 0, 2: None})          # swapped order
    assert out.splitlines()[0].startswith("1 ") and out.splitlines()[1].startswith("0 ")
    poly = "0 0.2 0.2 0.6 0.2 0.6 0.8 0.2 0.8\n"                            # polygon -> box
    out, kept, _ = convert_label_text(poly, {0: 0})
    x, y, w, h = [float(v) for v in out.split()[1:]]
    assert abs(x - 0.4) < 1e-6 and abs(y - 0.5) < 1e-6 and abs(w - 0.4) < 1e-6 and abs(h - 0.6) < 1e-6
    out, kept, _ = convert_label_text("0 1.2 -0.1 0.3 0.3\n", {0: 0})      # out-of-range centre is clamped
    assert all(0 <= float(v) <= 1 for v in out.split()[1:])

    # --- build two fake datasets in Roboflow layout
    tmp = tempfile.mkdtemp()

    def make(root, names, n_train, n_valid, label_fn):
        for split in ("train", "valid"):
            os.makedirs(f"{root}/{split}/images")
            os.makedirs(f"{root}/{split}/labels")
            for i in range(n_train if split == "train" else n_valid):
                cv2.imwrite(f"{root}/{split}/images/im{i}.jpg", np.full((32, 32, 3), i % 255, np.uint8))
                t = label_fn(i)
                if t is not None:
                    open(f"{root}/{split}/labels/im{i}.txt", "w").write(t)
        yaml.safe_dump({"names": names, "nc": len(names)}, open(f"{root}/data.yaml", "w"))

    # A: names lost ('0','1'); 60 train: 40 with fire, 10 with smoke, 10 empty; 20 valid like it
    A = f"{tmp}/A"
    make(A, ["0", "1"], 60, 20,
         lambda i: "0 0.5 0.5 0.3 0.3\n" if i % 3 else ("1 0.4 0.4 0.3 0.3\n" if i % 6 == 0 else ""))
    # B: D-Fire style: smoke=0, fire=1, object=2; 100 train: 20 fire, 30 only 'object' (-> negatives), 50 no label file
    B = f"{tmp}/B"
    make(B, ["smoke", "fire", "object"], 100, 10,
         lambda i: "1 0.5 0.5 0.2 0.2\n" if i < 20 else ("2 0.5 0.5 0.2 0.2\n" if i < 50 else None))

    sources = [
        dict(name="a", root=A, class_map={0: 0, 1: 1}, train_cap=None, val_cap=None),
        dict(name="b", root=B, class_map=None, train_cap=60, val_cap=None, max_empty_frac=0.5),
    ]
    out = os.path.join(tmp, "merged")
    stats = merge(sources, out, seed=3)
    a, b = stats["a"]["train"], stats["b"]["train"]
    assert a["images"] == 60 and a["empty"] == 10                                # nothing dropped without caps
    assert b["images"] == 60 and b["fire_boxes"] == 20 and b["smoke_boxes"] == 0
    assert b["empty"] == 40, b                                                  # 20 positives + 40 negatives (cap 60)
    assert b["dropped_boxes"] > 0                                               # 'object' boxes were dropped
    # B was auto-mapped from names: its 'fire' (id 1) must now be class 0
    lab = [f for f in glob.glob(f"{out}/train/labels/b__*.txt") if open(f).read().strip()]
    assert lab and all(open(f).read().split()[0] == "0" for f in lab)
    # file names carry the source prefix and every image has a label file
    ims = sorted(os.listdir(f"{out}/train/images"))
    lbs = sorted(os.listdir(f"{out}/train/labels"))
    assert all(n.startswith(("a__", "b__")) for n in ims)
    assert [os.path.splitext(i)[0] for i in ims] == [os.path.splitext(l)[0] for l in lbs]
    cfg = yaml.safe_load(open(f"{out}/data.yaml"))
    assert cfg["names"] == ["fire", "smoke"] and cfg["train"] == "train/images" and cfg["val"] == "valid/images"
    # a cap smaller than the positives is respected; empties never exceed their share
    stats2 = merge([dict(name="a", root=A, class_map={0: 0, 1: 1}, train_cap=30, val_cap=5,
                         max_empty_frac=0.2)], out, seed=3)
    s2 = stats2["a"]["train"]
    assert s2["images"] == 30 and s2["empty"] <= 6, s2
    assert stats2["a"]["valid"]["images"] == 5
    # unnamed classes without a hand mapping must be refused, not guessed
    try:
        merge([dict(name="a", root=A, class_map=None, train_cap=5, val_cap=5)], out)
        raise AssertionError("should have refused")
    except ValueError:
        pass
    # same seed -> same selection (reproducible)
    merge(sources, out, seed=5); first = sorted(os.listdir(f"{out}/train/images"))
    merge(sources, out, seed=5); assert first == sorted(os.listdir(f"{out}/train/images"))
    print_stats(stats)
    print("SELFTEST PASSED: class auto-mapping, polygon->box, clamping, negatives kept, caps, reproducibility, refusal to guess")


if __name__ == "__main__":
    _selftest()

## Step 4. Download the datasets
Paste your key. Each source has a **cap**: the most pictures taken from it, so one huge dataset cannot drown the others. `max_empty_frac` is the largest share of pictures *without* any fire/smoke that may be taken from that source.

- A source that fails to download is skipped with a message; the others still work.
- If a source says it has **no versions**, open it on Roboflow, fork it into your own workspace, generate a version, and change `workspace` / `project` below to your copy.
- To leave a source out, set `use=False`.

In [ ]:
ROBOFLOW_API_KEY = "PASTE_YOUR_KEY_HERE"   # <- your private key

SOURCES = [
    # FASDD lost its class names when exported: 0 = fire, 1 = smoke (checked on your v1 run).
    dict(name='fasdd', workspace='forestfiresmoke', project='fasdd_cv-dx83j', version=1,
         class_map={0: 0, 1: 1}, train_cap=12000, val_cap=1500, max_empty_frac=0.3, use=True),
    # D-Fire (Roboflow copy): many no-fire pictures; has an extra 'object' class that is dropped.
    dict(name='dfire', workspace='uweeaai', project='d-fire-ehbj2', version=None,
         class_map=None, train_cap=8000, val_cap=1000, max_empty_frac=0.4, use=True),
    # Small drone set: aerial view. Takes everything it has.
    dict(name='drone', workspace='tim-4ijf0', project='drone-fire-detection-byija', version=1,
         class_map=None, train_cap=None, val_cap=None, max_empty_frac=None, use=True),
]

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
for s in SOURCES:
    s['root'] = None
    if not s['use']:
        continue
    try:
        proj = rf.workspace(s['workspace']).project(s['project'])
        ver = s['version']
        if ver is None:
            nums = []
            for v in proj.versions():
                try: nums.append(int(str(v.version).split('/')[-1]))
                except ValueError: pass
            ver = max(nums) if nums else None
        if ver is None:
            raise RuntimeError('no versions: fork the dataset into your workspace and generate a version first')
        ds = proj.version(ver).download('yolov8')
        s['root'] = ds.location
        print('OK    ', s['name'], '->', ds.location)
    except Exception as e:
        print('FAILED', s['name'], ':', str(e)[:300])
ACTIVE = [s for s in SOURCES if s['root']]
print('\nusable sources:', [s['name'] for s in ACTIVE])

Expect `OK` lines with folder paths. If one says FAILED, read the message; the others continue.

## Step 5. Check each source's classes
Class numbers mean different things in different datasets, and a wrong mapping would teach the model that smoke is fire. This cell prints each source's class names and shows 8 pictures. **Orange = the source's class 0, blue = 1, green = 2, magenta = 3.**

- If the names are clear (`fire`, `smoke`, ...), the mapping is automatic and printed.
- If the names are just numbers, the cell says **NEEDS MANUAL MAPPING**. Look at the pictures, then fill `MANUAL` below. Example: orange boxes are on flames, blue on smoke: `MANUAL['drone'] = {0: 0, 1: 1}`. A class you do not want: `{0: 0, 1: 1, 2: None}`.
- Run the cell again after editing `MANUAL`.

In [ ]:
import matplotlib.pyplot as plt
import merge_datasets as md

MANUAL = {}        # e.g. MANUAL['drone'] = {0: 0}

for s in ACTIVE:
    names_ = md.read_names(s['root'])
    cmap = MANUAL.get(s['name']) or s['class_map'] or md.auto_map(names_)
    s['resolved_map'] = cmap
    print(f"\n=== {s['name']}: source classes {names_}")
    if cmap is None:
        print('    NEEDS MANUAL MAPPING (set MANUAL[%r] = {...} above and run again)' % s['name'])
    else:
        label = {0: 'fire', 1: 'smoke', None: 'DROPPED'}
        print('    mapping:', {names_[k] if k < len(names_) else k: label[v] for k, v in cmap.items()})
    fig = md.preview_grid(s['root'], cmap, n=8)
    plt.show()

**Look at the pictures.** On every source, the orange/blue colours must sit where you expect (flames vs smoke). If anything is swapped, fix `MANUAL` before going on. This is the single most important check in the notebook.

## Step 6. Build the merged training set

In [ ]:
for s in ACTIVE:
    if s.get('resolved_map') is None:
        raise SystemExit(f"{s['name']} still needs a manual class mapping (Step 5).")

stats = md.merge([dict(name=s['name'], root=s['root'], class_map=s['resolved_map'],
                       train_cap=s['train_cap'], val_cap=s['val_cap'],
                       max_empty_frac=s['max_empty_frac']) for s in ACTIVE],
                 MERGED_DIR, seed=1)
md.print_stats(stats)

tot_fire = sum(v['train']['fire_boxes'] for v in stats.values())
tot_smoke = sum(v['train']['smoke_boxes'] for v in stats.values())
tot_img = sum(v['train']['images'] for v in stats.values())
tot_empty = sum(v['train']['empty'] for v in stats.values())
print(f'\ntrain: {tot_img} pictures, {tot_empty} without fire/smoke ({100*tot_empty/max(tot_img,1):.0f}%), '
      f'{tot_fire} fire boxes, {tot_smoke} smoke boxes')
if tot_fire == 0: raise SystemExit('No fire boxes at all: the class mapping must be wrong.')
if tot_empty / max(tot_img, 1) > 0.6: print('WARNING: more than 60% empty pictures; lower max_empty_frac.')

Expect a table per source and split, and the totals. About 20 to 40 percent empty pictures is healthy. The `dropped` column counts boxes of classes we do not use (such as D-Fire's `object`).

## Step 7. Quick test run (about 5 to 10 minutes)
One round on 2% of the pictures. It proves the merged folder works before you spend an hour on the real run. The accuracy printed here is meaningless.

In [ ]:
from ultralytics import YOLO
START = V1_WEIGHTS if os.path.exists(V1_WEIGHTS) else 'yolov8n.pt'
print('starting from:', START)
YOLO(START).train(data=MERGED_YAML, epochs=1, imgsz=416, batch=16, fraction=0.02,
                  project=SAVE_DIR, name='v2_smoke_test', exist_ok=True, plots=False)

Expect training lines and a *Results saved to ...v2_smoke_test* message. If you see *out of memory*, change `batch=16` to `batch=8` here and in Step 8.

## Step 8. Fine-tune (the real run)
- Starts from your v1 model and trains up to 20 rounds at 416 px; stops early if validation stops improving.
- The learning-rate and optimiser settings are reasonable starting values, **not tuned**.
- The time depends on Colab. Watch the time of the first round and multiply by 20 to estimate; stop it and lower `epochs` if that is too long. Your v1 weights stay safe in Drive either way.

In [ ]:
from ultralytics import YOLO
model = YOLO(START)
model.train(data=MERGED_YAML, epochs=20, imgsz=416, batch=16, patience=8,
            optimizer='AdamW', lr0=0.001, warmup_epochs=1,
            project=SAVE_DIR, name=RUN_NAME, exist_ok=True)

**Resume cell** (only after a disconnect):

In [ ]:
from ultralytics import YOLO
YOLO(f'{SAVE_DIR}/{RUN_NAME}/weights/last.pt').train(resume=True)

## Step 9. Compare v1 and v2 on the same pictures
For every held-out picture the question is the one the drone asks: **did the detector say FIRE?** The table shows, per source and threshold, how many fire pictures it found (**recall**) and how many no-fire pictures (including smoke-only) it wrongly called fire (**false alarms**), for the old and the new model.

Reading it: v2 should be at least as good on FASDD and clearly better on D-Fire (v1 never saw D-Fire). If v2 gets worse on FASDD, tell Claude.

In [ ]:
import glob, numpy as np
from ultralytics import YOLO

V_IMG, V_LBL = f'{MERGED_DIR}/valid/images', f'{MERGED_DIR}/valid/labels'
files = sorted(glob.glob(V_IMG + '/*'))
srcs, kinds = [], []
for ip in files:
    stem = os.path.splitext(os.path.basename(ip))[0]
    classes = set()
    lp = f'{V_LBL}/{stem}.txt'
    if os.path.exists(lp):
        for line in open(lp):
            p = line.split()
            if p: classes.add(int(float(p[0])))
    srcs.append(stem.split('__')[0])
    kinds.append('fire' if 0 in classes else ('smoke_only' if classes else 'empty'))
srcs, kinds = np.array(srcs), np.array(kinds)
is_fire = kinds == 'fire'

def fire_scores(weights, imgsz=416, batch=32):
    m = YOLO(weights); out = []
    for i in range(0, len(files), batch):
        for r in m.predict(files[i:i+batch], imgsz=imgsz, conf=0.05, verbose=False):
            best = 0.0
            if r.boxes is not None and len(r.boxes):
                for c, p in zip(r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
                    if int(c) == 0: best = max(best, float(p))
            out.append(best)
    return np.array(out)

new = fire_scores(NEW_WEIGHTS)
old = fire_scores(V1_WEIGHTS) if os.path.exists(V1_WEIGHTS) else None
print(f'{len(files)} held-out pictures: {int(is_fire.sum())} with fire, {int((~is_fire).sum())} without\n')

def rate(mask, score, t):
    return float('nan') if mask.sum() == 0 else float((score[mask] >= t).mean())
fmt = lambda x: '  -  ' if x != x else f'{100*x:4.0f}%'
print(f"{'source':<8}{'fire n':>7}{'nofire n':>9}{'thr':>6}   {'recall old':>10}{'new':>6}   {'false alarm old':>15}{'new':>6}")
for sname in list(dict.fromkeys(srcs)) + ['ALL']:
    msk = np.ones(len(srcs), bool) if sname == 'ALL' else srcs == sname
    for t in (0.3, 0.4, 0.5):
        r_new, f_new = rate(msk & is_fire, new, t), rate(msk & ~is_fire, new, t)
        r_old = rate(msk & is_fire, old, t) if old is not None else float('nan')
        f_old = rate(msk & ~is_fire, old, t) if old is not None else float('nan')
        print(f"{sname:<8}{int((msk & is_fire).sum()):>7}{int((msk & ~is_fire).sum()):>9}{t:>6.1f}   {fmt(r_old):>10}{fmt(r_new):>6}   {fmt(f_old):>15}{fmt(f_new):>6}")
    print()

# threshold suggestion: the lowest threshold that keeps false alarms on no-fire pictures at or below 1%
SUGGESTED = None
for t in np.arange(0.25, 0.751, 0.05):
    if rate(~is_fire, new, t) <= 0.01:
        SUGGESTED = round(float(t), 2); break
print('Suggested per-picture threshold (false alarms <= 1% on these held-out pictures):', SUGGESTED)
print('This comes from the datasets only. Confirm it on your own no-fire clips (Step 10) before using it.')

# box-level numbers for the report
mv = YOLO(NEW_WEIGHTS).val(data=MERGED_YAML, imgsz=416, verbose=False)
METRICS = dict(map50=float(mv.box.map50), map50_95=float(mv.box.map),
               fire_precision=float(mv.box.p[0]), fire_recall=float(mv.box.r[0]),
               smoke_precision=float(mv.box.p[1]), smoke_recall=float(mv.box.r[1]))
print({k: round(v, 3) for k, v in METRICS.items()})

Write down the **ALL** rows and the box-level numbers for the review, and say they are measured on held-out pictures from the training datasets, not on drone footage.

## Step 10. Your own pictures and clips (optional, strongly recommended)
Upload `eval_images.py`, `eval_clips.py` and `yolo_fire_detector.py` (use the Files panel on the left), plus a folder `my_images` with subfolders `fire` and `nofire`. This is the honest test: lookalikes such as sunsets, orange objects, lamps and high-visibility jackets belong in `nofire`.

In [ ]:
import os
if os.path.isdir('my_images') and os.path.exists('eval_images.py'):
    !python eval_images.py --folder my_images --weights {NEW_WEIGHTS}
else:
    print('Skipped: upload eval_images.py, yolo_fire_detector.py and a my_images folder (fire/ and nofire/) first.')

In [ ]:
# video clips: needs clips.csv (path,fire_start_s,fire_end_s; leave the times empty for no-fire clips)
if os.path.exists('clips.csv') and os.path.exists('eval_clips.py'):
    !python eval_clips.py --manifest clips.csv --weights {NEW_WEIGHTS} --conf {SUGGESTED or 0.4}
else:
    print('Skipped: upload clips.csv, eval_clips.py and the clips first.')

## Step 11. Convert for the Raspberry Pi and save a record of this run

In [ ]:
import shutil, json, datetime
from ultralytics import YOLO
model = YOLO(NEW_WEIGHTS)
out = model.export(format='ncnn', imgsz=416)
shutil.make_archive(f'{SAVE_DIR}/fire_v2_ncnn_model', 'zip', out)
print('zip:', f'{SAVE_DIR}/fire_v2_ncnn_model.zip')

card = dict(date=str(datetime.date.today()), start_weights=START, run=RUN_NAME, imgsz=416,
            sources={s['name']: dict(workspace=s.get('workspace'), project=s.get('project'), license='CC BY 4.0 (check each page)') for s in ACTIVE},
            merged_stats=stats, held_out_metrics=globals().get('METRICS'),
            suggested_threshold=globals().get('SUGGESTED'),
            caveat='Held-out pictures come from the same datasets as training; real drone footage is untested here.')
json.dump(card, open(f'{SAVE_DIR}/{RUN_NAME}_model_card.json', 'w'), indent=2)
print('saved model card:', f'{SAVE_DIR}/{RUN_NAME}_model_card.json')

## Step 12. Live test with your laptop camera
Run after training. It prints a link ending in `.gradio.live`: open it in a **new tab**, click **Allow** for the camera, and show it a picture or video of fire on a phone or monitor. **Never use real fire.** The slider starts at the suggested threshold from Step 9.

In [ ]:
# ===== LIVE FIRE-DETECTION TEST UI (laptop camera) =====
# Run this cell AFTER training (it needs best.pt). It prints a link ending in .gradio.live.
# Open that link in a NEW TAB on your laptop and click "Allow" when the browser asks for the camera.
# TEST ONLY: show the camera a picture or video of fire on a phone or monitor. Never use real fire.
!pip install -q gradio

import time, cv2, numpy as np, gradio as gr
from ultralytics import YOLO

WEIGHTS = NEW_WEIGHTS   # made in Step 8
model = YOLO(WEIGHTS)

FIRE_ID = 0          # the merged dataset always has fire = 0, smoke = 1
CLASS_LABEL = {FIRE_ID: 'fire', 1 - FIRE_ID: 'smoke'}
CONFIRM_WINDOW, CONFIRM_COUNT = 10, 7          # same rule as the drone code: 7 of the last 10 frames

def detect(frame_bgr, conf_thr):
    """Returns a list of (class_id, confidence, x1, y1, x2, y2)."""
    r = model.predict(frame_bgr, imgsz=416, conf=conf_thr, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return []
    xyxy = r.boxes.xyxy.cpu().numpy(); cf = r.boxes.conf.cpu().numpy(); cl = r.boxes.cls.cpu().numpy()
    return [(int(c), float(p), *map(int, b)) for b, p, c in zip(xyxy, cf, cl)]

def process(frame, conf_thr, show_smoke, hist):
    if frame is None:
        return None, 'Waiting for the camera...', hist
    t0 = time.time()
    dets = detect(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR), conf_thr)
    out = frame.copy()
    fire_seen, lines = False, []
    for cls, c, x1, y1, x2, y2 in dets:
        is_fire = (cls == FIRE_ID)
        fire_seen = fire_seen or is_fire
        lines.append(f"{CLASS_LABEL.get(cls, cls)} {c:.2f}")
        if not is_fire and not show_smoke:
            continue
        col = (255, 90, 0) if is_fire else (60, 120, 255)
        cv2.rectangle(out, (x1, y1), (x2, y2), col, 3)
        cv2.putText(out, f"{CLASS_LABEL.get(cls, cls)} {c:.2f}", (x1 + 4, max(18, y1 + 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, col, 2)
    hist = (list(hist) + [1 if fire_seen else 0])[-CONFIRM_WINDOW:]
    n = sum(hist)
    ms = (time.time() - t0) * 1000
    if n >= CONFIRM_COUNT:
        head = f"## FIRE CONFIRMED ({n} of last {len(hist)} frames)\nThe drone would now wait for the pilot's confirm switch. Nothing is launched here."
    elif n > 0:
        head = f"## Checking... fire seen in {n} of last {len(hist)} frames (needs {CONFIRM_COUNT})"
    else:
        head = "## No fire"
    detail = ("Detections: " + ", ".join(lines)) if lines else "Detections: none"
    return out, f"{head}\n\n{detail}\n\nThis frame took {ms:.0f} ms on this computer. That is not the Raspberry Pi's speed.", hist

with gr.Blocks(title='Fire detection test') as demo:
    gr.Markdown("# Fire detection test\nShow the camera a picture or video of fire on a phone or monitor. **Do not use real fire.** "
                "Smoke is shown but never counts toward fire confirmation.")
    with gr.Row():
        with gr.Column():
            cam = gr.Image(sources=['webcam'], streaming=True, type='numpy', label='Camera in')
            conf = gr.Slider(0.1, 0.9, value=(SUGGESTED or 0.4), step=0.05, label='Confidence threshold (suggested from Step 9)')
            show_smoke = gr.Checkbox(True, label='Draw smoke boxes')
            reset = gr.Button('Reset the 7-of-10 counter')
        with gr.Column():
            out_img = gr.Image(label='What the detector sees', type='numpy')
            status = gr.Markdown('Waiting for the camera...')
    hist = gr.State([])
    cam.stream(process, [cam, conf, show_smoke, hist], [out_img, status, hist])
    reset.click(lambda: ([], '## Counter reset'), None, [hist, status])

demo.launch(share=True, debug=False)      # stop it later with: demo.close()

## Step 13. What to keep
In Drive, folder **fire_yolo**:
- `fire_v2_yolov8n/weights/best.pt` - the v2 model (use this with `--weights` in `fire_detector.py`).
- `fire_v2_ncnn_model.zip` - the Raspberry Pi version.
- `fire_v2_yolov8n_model_card.json` - what it was trained on and the numbers, for the report.

Then set the drone code's confidence threshold to the value you confirmed (`CONFIDENCE_THRESHOLD` in `yolo_fire_detector.py`).